# Week 5 · Notebook 1 — Regression
**CSCI 250 — Introduction to Artificial Intelligence · Fall 2026**

Predict a **number**. We fit linear regression on the **diabetes** dataset, read its error with **MAE / RMSE / R²**, add **Ridge** regularization, and plot predicted vs actual.

> Runs in Google Colab. No API keys needed. Part of **Assignment A5**.

## 0. Install

In [ ]:
!pip -q install scikit-learn matplotlib

## 1. Load the diabetes dataset
442 patients, 10 normalized health measurements, target = disease progression after one year.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print('train:', X_train.shape, ' test:', X_test.shape)
print('target range:', y.min(), '->', y.max())

## 2. Fit linear regression
Same `fit` / `predict` API as Week 4 — the target is just continuous now.

In [ ]:
from sklearn.linear_model import LinearRegression

reg = LinearRegression().fit(X_train, y_train)
preds = reg.predict(X_test)
# self-check: one prediction per test row
assert len(preds) == len(y_test)
print('first 5 predictions:', preds[:5].round(1))
print('first 5 actuals    :', y_test[:5])

## 3. Regression metrics
You can't use accuracy on numbers — measure **how far off** you are.
- **MAE**: average absolute miss (same units as y).
- **RMSE**: squares errors first, so big misses hurt more.
- **R²**: fraction of variance explained (1.0 = perfect).

In [ ]:
from sklearn.metrics import (mean_absolute_error,
    root_mean_squared_error, r2_score)

mae  = mean_absolute_error(y_test, preds)
rmse = root_mean_squared_error(y_test, preds)
r2   = r2_score(y_test, preds)
# self-checks: errors are non-negative and RMSE >= MAE always
assert mae >= 0 and rmse >= 0
assert rmse >= mae - 1e-9
print(f'MAE : {mae:.2f}')
print(f'RMSE: {rmse:.2f}   (>= MAE always)')
print(f'R^2 : {r2:.3f}')

## 4. Add regularization: Ridge
**Ridge** is linear regression that penalizes large weights (`alpha` = strength). On noisy data it often generalizes a little better. Compare the two.

In [ ]:
from sklearn.linear_model import Ridge

for alpha in [0.1, 1.0, 10.0]:
    rdg = Ridge(alpha=alpha).fit(X_train, y_train)
    p = rdg.predict(X_test)
    print(f'Ridge(alpha={alpha:>4}):  '
          f'RMSE={root_mean_squared_error(y_test, p):.2f}  '
          f'R^2={r2_score(y_test, p):.3f}')

## 5. Plot predicted vs actual
A perfect model lands on the diagonal. Spread around it = error.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5.5, 5.5))
plt.scatter(y_test, preds, alpha=0.6, edgecolor='k')
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, 'r--', label='perfect')
plt.xlabel('actual'); plt.ylabel('predicted')
plt.title('Diabetes — predicted vs actual'); plt.legend(); plt.show()

## 6. Reading the coefficients
`reg.coef_` holds **one weight per feature**. A coefficient is the change in the predicted target for a **one-unit increase in that feature, holding the others fixed**. Read it two ways:
- **Sign** — positive pushes progression **up**, negative pulls it **down**.
- **Magnitude** — bigger `|coef|` = stronger pull. The diabetes features are already normalized to a common scale, so the magnitudes are directly comparable; the largest `|coef|` is the most influential feature *for this model*.

Remember: this is **association, not causation**.

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes

names = load_diabetes().feature_names
order = np.argsort(np.abs(reg.coef_))[::-1]   # largest |weight| first
print('intercept:', round(reg.intercept_, 2))
for i in order:
    print(f'{names[i]:>4}: {reg.coef_[i]:+8.1f}')
# self-check: one coefficient per input feature
assert len(reg.coef_) == X_train.shape[1]
print('\nmost influential feature:', names[order[0]])

---
## A5 — Part 1 tasks
1. Report MAE, RMSE, and R² for plain LinearRegression (done above).
2. Pick the best `Ridge` alpha by RMSE and state whether it beat LinearRegression.
3. Print `reg.coef_` — which feature has the largest absolute weight?
4. Write 2–3 sentences interpreting the predicted-vs-actual plot.

In [ ]:
# Your A5 Part 1 work here
